In [1]:
import sys
sys.path.insert(0, "..")
import ccs_eeg_utils
from mne_bids import (BIDSPath, read_raw_bids)
import mne

bids_root = "../MNE-sample-data/ds006761"
subject_id = "01"

#print_dir_tree(bids_root, max_depth=4)

bids_path = BIDSPath(subject=subject_id, task="RPS",
                     datatype='eeg', suffix='eeg',
                     root=bids_root)

# read the file
raw = read_raw_bids(bids_path)
# fix the annotations readin
ccs_eeg_utils.read_annotations_core(bids_path,raw)

Extracting EDF parameters from c:\Users\bk57s\Visual Studio Code\EEG_Bala Sharks\MNE-sample-data\ds006761\sub-01\eeg\sub-01_task-RPS_eeg.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading events from ..\MNE-sample-data\ds006761\sub-01\eeg\sub-01_task-RPS_events.tsv.


C:\Users\bk57s\AppData\Local\Temp\ipykernel_25836\2920934692.py:17: RuntimeWarning: Did not find any channels.tsv associated with sub-01_task-RPS.

The search_str was "..\MNE-sample-data\ds006761\sub-01\**\eeg\sub-01*channels.tsv"
  raw = read_raw_bids(bids_path)


#### Re-referencing

#### Noisy-channels

#### Primary Method

In [ ]:
# Kanaltypen setzen (wie in milestone2). Falls Kanäle fehlen, wird der Fehler unterdrückt.
eeg_channels = [ch for ch in raw.ch_names if (ch.startswith('1-A') or ch.startswith('1-B') or ch.startswith('2-A') or ch.startswith('2-B'))]
eog_channels = [ch for ch in ['1-Erg1', '1-Erg2', '2-Erg1', '2-Erg2'] if ch in raw.ch_names]
resp_channels = [ch for ch in ['1-Resp', '2-Resp'] if ch in raw.ch_names]
bio_channels = [ch for ch in ['1-Plet', '2-Plet'] if ch in raw.ch_names]
temp_channels = [ch for ch in ['1-Temp', '2-Temp'] if ch in raw.ch_names]
stim_channels = [ch for ch in ['Status'] if ch in raw.ch_names]

if eeg_channels:
    raw.set_channel_types({ch: 'eeg' for ch in eeg_channels})
if eog_channels:
    raw.set_channel_types({ch: 'eog' for ch in eog_channels})
if resp_channels:
    raw.set_channel_types({ch: 'resp' for ch in resp_channels})
if bio_channels:
    raw.set_channel_types({ch: 'bio' for ch in bio_channels})
if temp_channels:
    raw.set_channel_types({ch: 'temperature' for ch in temp_channels})
if stim_channels:
    raw.set_channel_types({ch: 'stim' for ch in stim_channels})

# kurze Übersicht
print('Anzahl Kanäle total:', len(raw.ch_names))
print('Beispiel EEG-Kanäle:', eeg_channels[:10])

In [ ]:
# Re-referencing: zuerst linked-mastoid suchen, sonst bipolar-Fallback zwischen 1- and 2- Kanälen
def find_mastoid_candidates(ch_names):
    patterns = re.compile(r'(A1|A2|M1|M2|masto|mast|Masto|Mast)', re.IGNORECASE)
    return [ch for ch in ch_names if patterns.search(ch)]

mastoid_cands = find_mastoid_candidates(raw.ch_names)
raw_ref = None
if len(mastoid_cands) >= 2:
    ref_chs = mastoid_cands[:2]
    print('Applying linked-mastoid reference with:', ref_chs)
    # copy=True um original raw zu behalten
    raw_ref = mne.set_eeg_reference(raw.copy(), ref_channels=ref_chs, copy=True)
else:
    # Fallback: Bipolare Referenz zwischen homologen 1- vs 2- Kanälen
    pairs = []
    for ch in raw.ch_names:
        if ch.startswith('1-'):
            counterpart = ch.replace('1-', '2-')
            if counterpart in raw.ch_names:
                pairs.append((ch, counterpart))
    if pairs:
        print('No mastoids found — applying bipolar reference for', len(pairs), 'pairs')
        raw_ref = raw.copy()
        # Erzeuge für jedes Paar einen bipolar-kanal. Benenne als '1-ch-2-ch'.
        for a, b in pairs:
            ch_name = f'{a}-{b}'
            mne.set_bipolar_reference(raw_ref, anode=a, cathode=b, ch_name=ch_name, copy=False)
        # optional: drop original referenced channels oder belassen
    else:
        raise RuntimeError('Keine geeigneten Referenz-Kanäle gefunden. Bitte manuell angeben.')

In [ ]:
# Kurzer visueller Check (nicht blockierend beim automatischen Lauf):
print('Original EEG-Kanäle (erste 10):', raw.pick_types(eeg=True).ch_names[:10])
print('Referenzierte EEG-Kanäle (erste 10):', raw_ref.pick_types(eeg=True).ch_names[:10])
# Zum Plotten interaktiv: raw.plot(...) und raw_ref.plot(...)
# Beispiel (in Jupyter: block=True wenn gewünscht):
# raw.plot(picks='eeg', scalings={'eeg': 50e-6}, title='Original', show=True)
# raw_ref.plot(picks='eeg', scalings={'eeg': 50e-6}, title='Re-referenced', show=True)

# optional: speichern der neuen Raw-Datei
# raw_ref.save('sub-01_task-RPS_eeg_referenced_raw.fif', overwrite=True)